# Other Log Types

Samples module, parameter, buffer, and gradient-log types through safe introspection rather than heavyweight models.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.ModelLog.backward_peak_memory_bytes",
    "tl.ModelLog.backward_root_grad_fn_id",
    "tl.ModelLog.buffer_layers",
    "tl.ModelLog.buffer_num_passes",
    "tl.ModelLog.buffers",
    "tl.ModelLog.capture_cache_hit",
    "tl.ModelLog.capture_cache_key",
    "tl.ModelLog.capture_cache_path",
    "tl.ModelLog.capture_full_args",
    "tl.ModelLog.capture_kpis",
    "tl.ModelLog.check_metadata_invariants",
    "tl.ModelLog.cleanup",
    "tl.ModelLog.clear_hooks",
    "tl.ModelLog.conditional_arm_edges",
    "tl.ModelLog.conditional_branch_edges",
    "tl.ModelLog.conditional_edge_passes",
    "tl.ModelLog.conditional_elif_edges",
    "tl.ModelLog.conditional_else_edges",
    "tl.ModelLog.conditional_events",
    "tl.ModelLog.conditional_then_edges",
    "tl.ModelLog.current_function_call_barcode",
    "tl.ModelLog.detach_hooks",
    "tl.ModelLog.detach_saved_tensors",
    "tl.ModelLog.detect_loops",
    "tl.ModelLog.do",
    "tl.ModelLog.emit_nvtx",
    "tl.ModelLog.equivalent_operations",
    "tl.ModelLog.find_layers",
    "tl.ModelLog.find_sites",
    "tl.ModelLog.first_nonfinite",
    "tl.ModelLog.flops_by_type",
    "tl.ModelLog.fork",
    "tl.ModelLog.forward_lineno",
    "tl.ModelLog.grad_fn_logs",
    "tl.ModelLog.grad_fn_order",
    "tl.ModelLog.grad_fns",
    "tl.ModelLog.gradient_postfunc",
    "tl.ModelLog.gradient_postfunc_repr",
    "tl.ModelLog.gradients_to_save",
    "tl.ModelLog.graph_shape_hash",
    "tl.ModelLog.has_backward_log",
    "tl.ModelLog.has_conditional_branching",
    "tl.ModelLog.has_gradients",
    "tl.ModelLog.input_id_at_capture",
    "tl.ModelLog.input_layers",
    "tl.ModelLog.input_metadata",
    "tl.ModelLog.input_shape_hash",
    "tl.ModelLog.internally_initialized_layers",
    "tl.ModelLog.internally_terminated_bool_layers",
    "tl.ModelLog.internally_terminated_layers",
    "tl.ModelLog.intervention_ready",
    "tl.ModelLog.intervention_spec",
    "tl.ModelLog.io_format_version",
    "tl.ModelLog.is_appended",
    "tl.ModelLog.is_branching",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")